In [ ]:
import pandas as pd
from catboost import CatBoostClassifier, Pool
from itertools import product

In [ ]:
X_train = pd.read_parquet('data/X_train_half.parquet')
y_train = pd.read_parquet('data/y_train_half.parquet')
y_train = y_train['target']

In [ ]:
X_test = pd.read_parquet('data/X_test_half.parquet')
y_test = pd.read_parquet('data/y_test_half.parquet')
y_test = y_test['target']

In [ ]:
# Задаются категориальный признаки

cat_features = ['country', 'city', 'exp_group', 'os', 'source', 'topic', 'time_of_day']
cat_features_indices = [X_train.columns.get_loc(col) for col in cat_features]

In [ ]:
# Параметры грид серч заданы исходя из объема и сложности обучающего датасета

train_pool = Pool(X_train, y_train, cat_features=cat_features_indices)
test_pool = Pool(X_test, y_test, cat_features=cat_features_indices)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

grid = {
    'depth': [6, 8, 10],
    'learning_rate': [0.03, 0.05, 0.08],
    'l2_leaf_reg': [3, 5, 7]
}

keys = list(grid.keys())
all_params = list(product(*grid.values()))

results = []

for values in all_params:
    params = dict(zip(keys, values))

    model = CatBoostClassifier(
        **params,
        iterations=1000,
        loss_function='Logloss',
        eval_metric='AUC',
        scale_pos_weight=scale_pos_weight,
        grow_policy='SymmetricTree',    #Высокая скорость работы модели в проде
        task_type='GPU',
        verbose=100,
        od_type='Iter',
        od_wait=50,
        random_seed=10
    )

    model.fit(train_pool, eval_set=test_pool, plot=True)

    best_score = model.get_best_score()
    best_iter = model.get_best_iteration()

    row = params.copy()
    row['train_Logloss'] = best_score['learn']['Logloss']
    row['valid_Logloss'] = best_score['validation']['Logloss']
    row['valid_auc'] = best_score['validation']['AUC']
    row['best_iteration'] = best_iter
    row['gap'] = row['train_Logloss'] - row['valid_Logloss']
    results.append(row)

df_results = pd.DataFrame(results)
df_results = df_results.sort_values('valid_auc', ascending=False)

display(df_results.head(10))

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6236913	best: 0.6236913 (0)	total: 275ms	remaining: 4m 34s
100:	test: 0.6455293	best: 0.6455424 (99)	total: 23.4s	remaining: 3m 28s
200:	test: 0.6565807	best: 0.6565807 (200)	total: 50.6s	remaining: 3m 20s
300:	test: 0.6604563	best: 0.6604563 (300)	total: 1m 16s	remaining: 2m 56s
400:	test: 0.6631187	best: 0.6631187 (400)	total: 1m 41s	remaining: 2m 31s
500:	test: 0.6653404	best: 0.6653404 (500)	total: 2m 6s	remaining: 2m 5s
600:	test: 0.6671707	best: 0.6671772 (599)	total: 2m 31s	remaining: 1m 40s
700:	test: 0.6686394	best: 0.6686394 (700)	total: 2m 55s	remaining: 1m 15s
800:	test: 0.6698587	best: 0.6698587 (800)	total: 3m 20s	remaining: 49.8s
900:	test: 0.6709129	best: 0.6709129 (900)	total: 3m 45s	remaining: 24.8s
999:	test: 0.6718152	best: 0.6718152 (999)	total: 4m 9s	remaining: 0us
bestTest = 0.6718152165
bestIteration = 999


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Default metric period is 5 because AUC is/are not implemented for GPU


...

0:	test: 0.6263887	best: 0.6263887 (0)	total: 383ms	remaining: 6m 22s
100:	test: 0.6701565	best: 0.6702015 (99)	total: 51.6s	remaining: 7m 39s
200:	test: 0.6784230	best: 0.6784619 (199)	total: 1m 43s	remaining: 6m 52s
300:	test: 0.6804481	best: 0.6804692 (299)	total: 2m 37s	remaining: 6m 6s
400:	test: 0.6807965	best: 0.6808940 (376)	total: 3m 33s	remaining: 5m 18s
bestTest = 0.6808939576
bestIteration = 376
Shrink model to first 377 iterations.


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6263885	best: 0.6263885 (0)	total: 376ms	remaining: 6m 15s
100:	test: 0.6705826	best: 0.6705826 (100)	total: 50.9s	remaining: 7m 33s
200:	test: 0.6782973	best: 0.6783130 (199)	total: 1m 44s	remaining: 6m 53s
300:	test: 0.6808629	best: 0.6808629 (300)	total: 2m 38s	remaining: 6m 8s
400:	test: 0.6811005	best: 0.6811986 (368)	total: 3m 33s	remaining: 5m 19s
bestTest = 0.681198597
bestIteration = 368
Shrink model to first 369 iterations.


,depth,learning_rate,l2_leaf_reg,train_Logloss,valid_Logloss,valid_auc,best_iteration,gap
19,10,0.03,5,0.611423,0.635153,0.682166,994,-0.023730
18,10,0.03,3,0.610954,0.635223,0.682123,996,-0.024269
23,10,0.05,7,0.606268,0.635119,0.682003,684,-0.028851
20,10,0.03,7,0.612831,0.635291,0.681971,903,-0.022460
15,8,0.08,3,0.614855,0.635582,0.681936,973,-0.020727
22,10,0.05,5,0.602147,0.635122,0.681879,774,-0.032975
17,8,0.08,7,0.615915,0.635479,0.681851,916,-0.019563
16,8,0.08,5,0.616090,0.635586,0.681784,899,-0.019496
21,10,0.05,3,0.606442,0.635351,0.681672,649,-0.028909
26,10,0.08,7,0.609206,0.635598,0.681199,368,-0.026392


In [ ]:
df_results

,depth,learning_rate,l2_leaf_reg,train_Logloss,valid_Logloss,valid_auc,best_iteration,gap
19,10,0.03,5,0.611423,0.635153,0.682166,994,-0.023730
18,10,0.03,3,0.610954,0.635223,0.682123,996,-0.024269
23,10,0.05,7,0.606268,0.635119,0.682003,684,-0.028851
20,10,0.03,7,0.612831,0.635291,0.681971,903,-0.022460
15,8,0.08,3,0.614855,0.635582,0.681936,973,-0.020727
22,10,0.05,5,0.602147,0.635122,0.681879,774,-0.032975
17,8,0.08,7,0.615915,0.635479,0.681851,916,-0.019563
16,8,0.08,5,0.616090,0.635586,0.681784,899,-0.019496
21,10,0.05,3,0.606442,0.635351,0.681672,649,-0.028909
26,10,0.08,7,0.609206,0.635598,0.681199,368,-0.026392


## Выводы
Выбрана одна модель для дальнейщего обучения:
№ 14 (8, 0.05 и 7) Т.к. у неё valid_auc близок к максимальной, но низкий gap. Т.е. данная модель обладает хорошей обобщающей способностью